# 🔗 Phase 3 — Multimodal Late Fusion (Visual CNN + Agronomic Metadata)

In this notebook, we combine leaf imagery with agronomic field metadata using **Multimodal Late Fusion**:
- **Visual Branch:** EfficientNet-B3 pre-trained feature extractor producing a 256-d dense embedding.
- **Agronomic Metadata Branch:** Dense(32) + BatchNormalization encoding crop variety, seed batch, and planting season.
- **Fusion Head:** Concatenation (288-d) -> Dense(128) -> Dropout(0.3) -> Softmax(4 classes).
- **Why Fusion?** Certain maize cultivars (e.g. *SAMMAZ 15*) possess bred genetic resistance to specific blights, providing valuable inductive prior when visual symptoms are ambiguous.


In [1]:
# Setup & Imports
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path(".").resolve().name == "notebooks" else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf

from src.phase1_cnn.data_pipeline import CLASS_NAMES, IMG_SIZE
from src.phase2_ocr.encoder import encode, METADATA_DIM
from src.phase3_fusion.fusion_model import build_fusion_model

print("TensorFlow:", tf.__version__)
print("Metadata Dimension:", METADATA_DIM)


TensorFlow: 2.16.2
Metadata Dimension: 17


### 1. Build and Inspect the Multimodal Fusion Model

In [2]:
# Build fusion model with Phase 1 CNN weights
CNN_WEIGHTS = "models/checkpoints/phase1_stage2_best.keras"
fusion_model = build_fusion_model(cnn_weights_path=CNN_WEIGHTS, freeze_cnn=True)

print("=" * 65)
print("MULTIMODAL FUSION ARCHITECTURE SUMMARY:")
print("=" * 65)
for layer in fusion_model.layers:
    trainable_str = "Trainable" if layer.trainable else "Frozen"
    print(f"{layer.name:<24} | Output: {str(layer.output.shape):<22} | {trainable_str}")
print("=" * 65)


Loaded CNN model from models/checkpoints/phase1_stage2_best.keras
MULTIMODAL FUSION ARCHITECTURE SUMMARY:
metadata_input           | Output: (None, 17)             | Trainable
image_input              | Output: (None, 300, 300, 3)    | Trainable
meta_dense               | Output: (None, 32)             | Trainable
feature_extractor        | Output: (None, 256)            | Trainable
meta_bn                  | Output: (None, 32)             | Trainable
fusion_concat            | Output: (None, 288)            | Trainable
fusion_dense_128         | Output: (None, 128)            | Trainable
fusion_dropout           | Output: (None, 128)            | Trainable
predictions              | Output: (None, 4)              | Trainable


### 2. Dual-Input Inference Demonstration

In [3]:
# Load a real leaf sample
test_img_path = "data/raw/plantvillage/data/Common_Rust/Corn_Common_Rust (921).JPG"
img = Image.open(test_img_path).resize(IMG_SIZE)
img_arr = np.expand_dims(np.array(img, dtype=np.float32), 0)

# Simulate two distinct agricultural field contexts for the exact same leaf:
# Context A: Resistant variety planted in dry conditions
meta_a = encode({
    "crop_variety": "SAMMAZ 15",
    "batch_number": "LOT-2024-001",
    "planting_date": "2024-06-01"
})

# Context B: Highly susceptible variety planted in peak humid season
meta_b = encode({
    "crop_variety": "OBA SUPER 2",
    "batch_number": "LOT-2023-088",
    "planting_date": "2023-11-01"
})

meta_batch_a = np.expand_dims(meta_a, 0)
meta_batch_b = np.expand_dims(meta_b, 0)

pred_a = fusion_model.predict([img_arr, meta_batch_a], verbose=0)[0]
pred_b = fusion_model.predict([img_arr, meta_batch_b], verbose=0)[0]

# Plot Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.barh(CLASS_NAMES, pred_a, color="#3498db")
ax1.set_title("Prediction with Context A (SAMMAZ 15)", fontsize=11)
ax1.set_xlim(0, 1.05)
for i, v in enumerate(pred_a):
    ax1.text(v + 0.02, i, f"{v:.1%}", va="center")

ax2.barh(CLASS_NAMES, pred_b, color="#e67e22")
ax2.set_title("Prediction with Context B (OBA SUPER 2)", fontsize=11)
ax2.set_xlim(0, 1.05)
for i, v in enumerate(pred_b):
    ax2.text(v + 0.02, i, f"{v:.1%}", va="center")

plt.suptitle("Multimodal Fusion Inference on Identical Leaf", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


<Figure size 1400x400 with 2 Axes>